In [51]:
import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
from utils import *

In [7]:
f_path = root_path + 'MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/'
# Read files with appropriate template RECON2s reactions and remove "\n"
f = open(f_path + "rxnFormula_HUMAN.txt")
rxnFormula = f.read().splitlines()
f.close()

f = open(f_path + "rxnAbbreviation_HUMAN.txt")
rxnAbbreviation = f.read().splitlines()
f.close()

f = open(f_path + "rxnPathway_HUMAN.txt")
rxnPathway = f.read().splitlines()
f.close()

f = open(f_path + "rxnConditions_HUMAN.txt")
rxnConditions = f.read().splitlines()
f.close()

f = open(f_path + "rxnGPRs_HUMAN.txt")
rxnGPR = f.read().splitlines()
f.close()

# Load PSIM matrix
f = open(f_path + 'PSIM_HUMAN.tab','r')
PSIM=f.read().splitlines()
f.close()

#Extract the Uniprot IDs
PSIM_entries = []
for line in PSIM:
    PSIM_entries.append(line.split('\t')[0])

	# Define Basic Functions

#Define a functions that counts of amino acids
def count_AAs(sequence):
    AAcounts = []    
    AAs = ['G', 'A', 'V', 'L', 'I', 'M', 'W', 'F', 'P', 'S', 'T', 'C', 'Y', 'N', 'Q', 'E', 'D', 'K', 'R', 'H']
    for aa in AAs:
        AAcounts.append(sequence.count(aa))
    return AAcounts

#Define a function that substitutes question marks for AA counts in a formula (string)
#Useful for translation and protein degradation pathway

def substitute_AAs_count(formula,AAcounts):
    template = "? gly[c] + ? ala_L[c] + ? val_L[c] + ? leu_L[c] + ? ile_L[c] + ? met_L[c] + ? trp_L[c] + ? phe_L[c] + ? pro_L[c] + ? ser_L[c] + ? thr_L[c] + ? cys_L[c] + ? tyr_L[c] + ? asn_L[c] + ? gln_L[c] + ? glu_L[c] + ? asp_L[c] + ? lys_L[c] + ? arg_L[c] + ? his_L[c]"
    new = ''
    index = 0
    for character in template:
        if character == "?":
            new = new + str(AAcounts[index])
            index = index + 1
        else:
            new = new + character
    newFormula = formula.replace(template,new)
    return newFormula

# Function that generates the translation reaction of a protein given its UniProt ID
def translate_protein(entryID):
    #Obtain protein sequence and length
    PSI_row = PSIM[PSIM_entries.index(str(entryID))] 
    PSI_row = PSI_row.split('\t')
    sequence = PSI_row[11]
    AAcounts = count_AAs(sequence)
    N = len(sequence) #Number to replace atp's, gtp's, ppi's, pi's, h2o's, amp's, and gpd's
    templateFormula = "? h2o[c] + ? atp[c] + ? gtp[c] + ? gly[c] + ? ala_L[c] + ? val_L[c] + ? leu_L[c] + ? ile_L[c] + ? met_L[c] + ? trp_L[c] + ? phe_L[c] + ? pro_L[c] + ? ser_L[c] + ? thr_L[c] + ? cys_L[c] + ? tyr_L[c] + ? asn_L[c] + ? gln_L[c] + ? glu_L[c] + ? asp_L[c] + ? lys_L[c] + ? arg_L[c] + ? his_L[c] -> ? h[c] + ? amp[c] + adp[c] + ? pi[c] + ? gdp[c] + ? ppi[c] + XXX[c]"
    translationFormula = substitute_AAs_count(templateFormula, AAcounts)
    translationFormula = translationFormula.replace("? h2o[c]",str(2*N-1)+" h2o[c]")
    translationFormula = translationFormula.replace("? h[c]",str(2*N-1)+" h[c]")
    translationFormula = translationFormula.replace("? ppi[c]",str(N)+" ppi[c]")
    translationFormula = translationFormula.replace("? pi[c]",str(2*N-1)+" pi[c]")
    translationFormula = translationFormula.replace("? gdp[c]",str(2*N-2)+" gdp[c]")
    translationFormula = translationFormula.replace("? amp[c]",str(N)+" amp[c]")
    translationFormula = translationFormula.replace("? gtp[c]",str(2*N-2)+" gtp[c]")
    translationFormula = translationFormula.replace("? atp[c]",str(N+1)+" atp[c]")
    translationFormula = translationFormula.replace("XXX[c]",str(entryID)+"[c]")
    return translationFormula

# Function that replaces the XXX template name for the UniProt ID
def insert_prot_name_in_rxnFormula(formula,entryID):
    newFormula = formula.replace("XXX",str(entryID))
    return newFormula

# Function that replaces the XXX template name for the Reaction abbreviation
def insert_prot_name_in_rxnName(rxnAbbrev,entryID):
    newAbbreviation = str(entryID) + "_" + rxnAbbrev
    return newAbbreviation

#Function for adding the reactions of a given PathwayName
def addPathway(pathwayName,listOfRxns,listOfRxnsNames):   
    newList = listOfRxns    
    newList2 = listOfRxnsNames
    for i in range(len(rxnPathway)):
        if rxnPathway[i] == pathwayName:
            newList.append(rxnFormula[i])
            newList2.append(rxnAbbreviation[i])
    return newList,newList2

#Function for adding the reactions of a given PathwayName given a condition
def addPathwayFromCondition(conditionName,listOfRxns,listOfRxnsNames):   
    newList = listOfRxns 
    newList2 = listOfRxnsNames
    for i in range(len(rxnConditions)):
        if rxnConditions[i] == conditionName:
            newList.append(rxnFormula[i])
            newList2.append(rxnAbbreviation[i])
    return newList,newList2

#Function for creating a list of GPRs given a list of Rxn names
def getGPRsFromRxnNames(listOfRxnsNames):
    GPR_list = []
    for reaction in listOfRxnsNames:
        if reaction in rxnAbbreviation:
            GPR_list.append(rxnGPR[rxnAbbreviation.index(reaction)])
        else:
            GPR_list.append('')
    return GPR_list
     
#Function that adds canonical reactions to overall list
def addCanonicalRxns(listOfRxns,listOfRxnNames,listOfGPRs):
    newListRxns = listOfRxns
    newListNames = listOfRxnNames
    newListGPRs = listOfGPRs
    #Add  canonical reactions
    [newListRxns, newListNames] = addPathway("Canonical",listOfRxns,listOfRxnNames)
    #Add canonical GPRs
    r = []
    n = []
    [r,n] = addPathway("Canonical",r,n)
    newGPRs = getGPRsFromRxnNames(n)
    for gpr in newGPRs:
        newListGPRs.append(gpr)
    return newListRxns, newListNames, newListGPRs

In [25]:
import pandas as pd
d = pd.read_csv(f_path + 'PSIM_HUMAN.tab',sep = '\t')
d.iloc[2,[0,4,5,6,7,8,9,10]]

Entry       P06865
SP               1
DSB              3
GPI              0
NG               3
OG               0
TMD              0
Location       [l]
Name: 2, dtype: object

In [179]:
entryID = 'P06865'
PSI_row = PSIM[PSIM_entries.index(str(entryID))] 
PSI_row = PSI_row.split('\t')
sequence = PSI_row[11]
L = float(PSI_row[2]) # Protein Length
MW = float(PSI_row[3]) # Molecular weight
PSI = []
Kv = 0.7
V = MW * 1.21 / 1000.0 # Protein Volume in nm^3
clathrin_coeff = int(round(29880.01 * Kv / V)) # Number of proteins per clathrin vesicle  
copi_coeff = int(round(143793.19 * Kv / V))
copii_coeff = int(round(268082.35 * Kv / V))
connector = ''
#Prepare vectors that will store reactions and components
protName = str(entryID)
rxns = []
rxnNames = []   

#Add translation reaction
translation_reaction = translate_protein(protName)
rxns.append(translation_reaction)
rxnNames.append("TRANSLATION_protein") 
for i in [0,4,5,6,7,8,9,10]:
    PSI.append(PSI_row[i])  #This is the vector from the PSIM that corresponds to the given protein 
                                      #[entry,SP,DSB,GPI,NG,OG,TMD,SubCellLoc] 

if PSI[1] == '0': #If it doesn't have signal peptide then ignore
    raise ValueError('NAAWWWW')

elif PSI[1] == '1': #Translocate protein if it has signal peptide
    if L <= 160:
        [rxns,rxnNames] = addPathway("Post-translational Translocation",rxns,rxnNames)
        if PSI[7] == '[e]' or PSI[7] == '':
            [rxns,rxnNames] = addPathway("Post-translational Translocation (Secretory protein)",rxns,rxnNames)
        if PSI[7] == "[pm]" or PSI[6] != '0':
            [rxns,rxnNames] = addPathway("Post-translational Translocation (Tail anchored membrane protein)",rxns,rxnNames)
    else:
        [rxns,rxnNames] = addPathway("Translocation",rxns,rxnNames)

    number_BiP = L/40 #Number of BiPs depends on protein length http://www.cshperspectives.com/content/5/5/a013201.full
    for i in range(len(rxns)):
        rxns[i] = rxns[i].replace("!",str(number_BiP))
    connector = 'XXX[r]'

In [180]:
#Add GPI reactions

rxns.append(connector + ' -> XXX_preGPI[r]')
rxnNames.append('Start_GPI')
[rxns,rxnNames] = addPathwayFromCondition('GPI=1',rxns,rxnNames)
connector = 'XXX-dgpi_hs[r]'

In [181]:
GPRs = getGPRsFromRxnNames(rxnNames)
for i in range(len(rxns)):
    rxns[i] = insert_prot_name_in_rxnFormula(rxns[i],protName)
for i in range(len(rxnNames)):
    rxnNames[i] = insert_prot_name_in_rxnName(rxnNames[i],protName)

In [183]:
rxns[-3:]

['P06865[r] -> P06865_preGPI[r]',
 'P06865_preGPI[r] + gpi_hs[r] -> P06865-gpi_hs[r] + gpi_sig[r]',
 'P06865-gpi_hs[r] + h2o[r] -> P06865-dgpi_hs[r] + hdca[r] + h[r]']

In [184]:
GPRs[-3:]

['', '(10026) and (51604) and (94005) and (128869) and (8733)', '(80055)']